# Parte 1

In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

#spark = SparkSession.builder.appName("BotNet").getOrCreate()
sc = SparkContext("local[*]", "BotNet1")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 11:51:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 11:51:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import numpy as np
import time 

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


In [ ]:
import numpy as np
import random
from tqdm.notebook import trange, tqdm
from copy import deepcopy

def readFile (filename): 
    """
    Arguments: 
        filename – name of the spam dataset file 
        12 columns: 11 features/dimensions (X) + 1 column with labels (Y) 
        Y -- Train labels (0 if normal traffic, 1 if botnet)  
        m rows: number of examples (m) 
    Returns: 
        An RDD containing the data of filename. Each example (row) of the file 
        corresponds to one RDD record. Each record of the RDD is a tuple (X,y).  
        “X” is an array containing the 11 features (float number) of an example  
        “y” is the 12th column of an example (integer 0/1)  
    """
    '''
    data = sc.textFile(filename)  

    def clasiffication(line:str):
        words = line.split(',')

        X = [float(word) for word in words[:-1]]
        Y = int(words[-1]) 

        return np.array(X), Y
    
    data_rdd = data.map(clasiffication)

    return data_rdd
    '''

    return sc.textFile(filename).map(lambda line: np.array(line.split(","), dtype=float)).map(lambda x: (x[:-1], x[-1]))



def normalize (RDD_Xy): 
    """
    Arguments: 
        RDD_Xy is an RDD containing data examples. Each record of the RDD is a tuple 
        (X,y). 
        “X” is an array containing the 11 features (float number) of an example 
        “y” is the label of the example (integer 0/1)  
    Returns: 
        An RDD rescaled to N(0,1) in each column (mean=0, standard deviation=1) 
    """
    stats = RDD_Xy.map(lambda x: (x[0], x[0] ** 2, 1)).reduce(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))
    mean = stats[0]/stats[2]
    std = np.sqrt((stats[1] - stats[1]/stats[2])/stats[2])
    rdd = RDD_Xy.map(lambda xy: ((xy[0] - mean)/std, xy[1]))

    '''
    mean_rdd = RDD_Xy.map(lambda x: ("key", (x[0], 1, x[0] ** 2))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2])).mapValues(lambda x: (x[0] / x[1], x[2] / x[1]))
    stats = mean_rdd.first()[1]
    mean = stats[0]
    var = stats[1] - (mean ** 2)
    std = np.sqrt(var)
    rdd = RDD_Xy.map(lambda xy: ((xy[0] - mean)/std, xy[1]))
    '''
    '''
    n_rows = RDD_Xy.count()

    sum = RDD_Xy.map(lambda xy: xy[0]).reduce(lambda a, b: a + b)
    mean = sum/n_rows

    sum_std = RDD_Xy.map(lambda xy: (xy[0] - mean)**2).reduce(lambda a, b: a + b)
    std = np.sqrt(sum_std/n_rows)

    std[std==0] = 1.0
    print(std)

    rdd = RDD_Xy.map(lambda xy: ((xy[0] - mean)/std, xy[1]))
    '''
    
    return rdd


def train (RDD_Xy, iterations, learning_rate, lambda_reg=1, print_cost=False): 
    """
    Arguments: 
        RDD_Xy --- RDD containing data examples. Each record of the RDD is a tuple 
        (X,y). 
        “X” is an array containing the 11 features (float number) of an example 
        “y” is the label of the example (integer 0/1)  
        iterations -- number of iterations of the optimization loop 
        learning_rate -- learning rate of the gradient descent 
        lambda_reg –- regularization rate 
    Returns: 
        A list or array containing the weights “w” and bias “b”	at the end of the 
        training process	
    """
    n_rows = RDD_Xy.count()
    n_cols = len(RDD_Xy.first()[0])
    #n_cols = len(RDD_Xy.map(lambda xy: xy[0]).reduce(lambda a, b: a))

    ws = np.random.random(n_cols)
    b = np.random.random()

    for _ in tqdm(range(iterations)):
        # X, y, pred
        rdd_preds = RDD_Xy.map(lambda xy: (
            xy[0],
            xy[1],
            sigmoid(np.dot(ws, xy[0]) + b)
        ))
        #rdd_preds.cache()

        # Tamaño del paso
        grads = rdd_preds.map(
            lambda xyp: ((xyp[2] - xyp[1]) * xyp[0], xyp[2] - xyp[1])
        ).reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))

        dws = grads[0] / n_rows + lambda_reg / n_cols * ws
        db = grads[1] / n_rows

        if print_cost:
            cost = rdd_preds.map(lambda xyp:
                xyp[1] * np.log(xyp[2] + 1e-15) +
                (1 - xyp[1]) * np.log(1 - xyp[2] + 1e-15)
            ).reduce(lambda a, b: a + b)
            cost = -cost / n_rows + (lambda_reg / (2 * n_cols)) * np.dot(ws, ws)
            tqdm.write(f"Iteration cost: {cost:.6f}")

        # Actualizar los pesos
        ws = ws - learning_rate* dws
        b = b - learning_rate * db
        #rdd_preds.unpersist()
        '''
        #print(f"Acuracy: {accuracy(ws, b, RDD_Xy)}")
        rdd_preds = RDD_Xy.map(lambda xy: (xy[0], xy[1], predict(ws, b, xy[0])))
        rdd_preds.cache()

        # Tamaño del paso
        dws = rdd_preds.map(lambda xyp: ("key", ((xyp[2] - xyp[1]) * xyp[0], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda x: x[0] / x[1]).first()[1]
        dws += lambda_reg/n_cols * ws

        db = rdd_preds.map(lambda xyp: ("key", (xyp[2] - xyp[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda x: x[0] / x[1]).first()[1]

        # Actualizar los pesos
        ws = ws - learning_rate * dws
        b = b - learning_rate * db
        '''
    return ws, b


def predict (w, b, X): 
    """
    Arguments: 
        w -- weights 
        b -- bias 
        X –- Example to be predicted  
    Returns: 
        Y_pred –- a value (0/1) corresponding to the prediction of X  
    """
    prob = sigmoid(np.dot(w, X) + b)

    return 1 if prob >= 0.5 else 0


def accuracy (w, b, RDD_Xy): 
    """
    Arguments: 
        w -- weights 
        b -- bias 
        RDD_Xy –- RDD containing examples to be predicted  
    Returns: 
        accuracy -- the number of predictions that are correct divided by the number         
        of records (examples) in RDD_xy.  
        Predict function can be used for predicting a single example
    """
    #m = RDD_Xy.count()

    #preds = RDD_Xy.map(lambda xy: xy[1] == predict(w, b, xy[0])).reduce(lambda a, b: a + b)

    #preds = RDD_Xy.map(lambda xy: ("key",(xy[1] == predict(w, b, xy[0]), 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda x: x[0] / x[1]).first()[1]
    
    correct, total = (
        RDD_Xy
        .map(lambda xy: (int(xy[1] == predict(w, b, xy[0])), 1))
        .reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    )
    preds = (correct / total)
    
    return preds * 100  

In [14]:
path = "/opt/local/practica_spark/datos/botnet_tot_syn_l.csv"
nIter = 10
learningRate = 1e-4
lambda_reg = 1

# read data
data = readFile(path)

data.cache()

# standardize
data=normalize(data)
end = time.time()

#
ws, b = train(data, nIter, learningRate, lambda_reg=lambda_reg, print_cost=True)
acc = accuracy(ws, b, data)
print (f"acc: {acc:.2f}")

  0%|          | 0/10 [00:00<?, ?it/s]

Iteration cost: 1.544492


Iteration cost: 1.544441


Iteration cost: 1.544391
Iteration cost: 1.544340
Iteration cost: 1.544289
Iteration cost: 1.544239


Iteration cost: 1.544188


Iteration cost: 1.544138
Iteration cost: 1.544087


Iteration cost: 1.544036
acc: 34.54


In [5]:
print(f"Pesos: {ws}")
print(f"Bias: {b}")

Pesos: [0.79502946 0.51483186 0.10581514 0.06416657 0.43843893 0.75422907
 0.33889227 0.43528462 0.79291163 0.2872463  0.20472048]
Bias: 0.6099081160533865
